# Chung khoan SSI
# Mot chien luoc mua: Khi MA10 > MA20 va Gia du doan > Gia thuc te
# Mot chien luoc ban: Khi MA10 < MA20 va gia du doan < Gia thuc te

# Load data

In [ ]:
import sys
sys.path.append("../Common")
import CommonSSI

# Process ML, Data

In [ ]:
from datetime import datetime, timedelta
import talib
from sklearn import linear_model
import redis

# Step 1----Load data
# Process ML, Data
symbol = "ACB"
from_date = (datetime.now() - timedelta(days=300)).strftime("%Y-%m-%d") # "2025-10-01"
to_date = datetime.now().strftime("%Y-%m-%d") # "2025-10-24"

data = CommonSSI.CommonSSI.loaddataSSI_Ext(symbol, from_date, to_date)

# Step 2----Process ML, data
# Chuyen Close thanh float
data["Close"] = data["Close"].astype(float)

data["MA10"] = talib.SMA(data["Close"], timeperiod=10)
data["MA20"] = talib.SMA(data["Close"], timeperiod=20)

# Gia dong cua ngay hom qua
X = data[["Close"]].shift(1).fillna(0)
Y = data[["Close"]]

model = linear_model.LinearRegression()
model.fit(X, Y)

data["Predict_Close"] = model.predict(X)

data["Buy_Signal"] = (data["MA10"] > data["MA20"]) & (data["Predict_Close"] > data["Close"])
data["Sell_Signal"] = (data["MA10"] < data["MA20"]) & (data["Predict_Close"] < data["Close"])

# Step 3----Day sang Redis

    Symbol Market TradingDate  Time   Open   High    Low  Close    Volume  \
0      ACB   HOSE  30/12/2024  None  21180  21431  21180  21222   5000400   
1      ACB   HOSE  31/12/2024  None  21222  21556  21180  21556   8841000   
2      ACB   HOSE  02/01/2025  None  21389  21472  21222  21389   4567900   
3      ACB   HOSE  03/01/2025  None  21305  21347  20971  20971   5017300   
4      ACB   HOSE  06/01/2025  None  20888  20929  20679  20720  12052700   
..     ...    ...         ...   ...    ...    ...    ...    ...       ...   
198    ACB   HOSE  20/10/2025  None  25600  25650  24600  24800  32231800   
199    ACB   HOSE  21/10/2025  None  24850  25300  24850  25000  19371400   
200    ACB   HOSE  22/10/2025  None  25200  25300  24650  25100  15620500   
201    ACB   HOSE  23/10/2025  None  25100  25200  24950  24950   8751200   
202    ACB   HOSE  24/10/2025  None  24950  25300  24800  25000  11724500   

                 Value  
0    127282210000.0010  
1         226672915000  


,Datetime,Open,High,Low,Close,Volume,MA10,MA20,Predict_Close,Buy_Signal,Sell_Signal
0,2024-12-30,21180,21431,21180,21222.0,5000400,NaN,NaN,6947.612835,False,False
1,2024-12-31,21222,21556,21180,21556.0,8841000,NaN,NaN,21757.359891,False,False
2,2025-01-02,21389,21472,21222,21389.0,4567900,NaN,NaN,21990.441387,False,False
3,2025-01-03,21305,21347,20971,20971.0,5017300,NaN,NaN,21873.900639,False,False
4,2025-01-06,20888,20929,20679,20720.0,12052700,NaN,NaN,21582.199844,False,False
...,...,...,...,...,...,...,...,...,...,...,...
198,2025-10-20,25600,25650,24600,24800.0,32231800,26330.0,26052.5,24917.219219,True,False
199,2025-10-21,24850,25300,24850,25000.0,19371400,26185.0,26035.0,24254.262867,False,False
200,2025-10-22,25200,25300,24650,25100.0,15620500,26020.0,25990.0,24393.832625,False,False
201,2025-10-23,25100,25200,24950,24950.0,8751200,25820.0,25955.0,24463.617505,False,True


# Cho chay tu dong